In [1]:
import glob
from datetime import datetime
import numpy as np
from matplotlib import pyplot as plt

In [2]:
sweep_pattern = "/gpfs/projects/kernlab/murillor/analysis2/logs/sweep/*"

sweep_paths = glob.glob(sweep_pattern)

bgs_pattern = "/gpfs/projects/kernlab/murillor/analysis2/logs/bgs/*"

bgs_paths = glob.glob(bgs_pattern)

neutral_pattern = "/gpfs/projects/kernlab/murillor/analysis2/logs/neutral/*"

neutral_paths = glob.glob(neutral_pattern)

In [ ]:
len(sweep_paths)

In [4]:
def get_runtimes(file_paths):
    """
    Get the runtimes in the Snakemake logs for a list of paths.
    Returns a list of runtimes in minutes.
    """
    diff_min_list = {"Empty": []}
    for path in file_paths:
        diff_min = np.nan
        id_str = "Empty"
        fh = open(path, mode="r")
        times_list = []
        finished = False
        for line in fh:
            if not line.startswith("["):
                if "Finished" in line:
                    finished = True
                if "wildcards" in line:
                   id_str = line.split("wildcards: ")[1].split("seed=")[0].strip()
                   if id_str not in diff_min_list:
                       diff_min_list[id_str] = []
                continue
            line = line.strip()
            date_str = line[1:-1]
            times_list.append(datetime.strptime(date_str, "%a %b %d %H:%M:%S %Y"))
        if len(times_list) != 2:
            print(f"Log has {len(times_list)} timestamps, skipping")
        if finished:
            diff_min = (times_list[-1] - times_list[0]).total_seconds()/60
        diff_min_list[id_str].append(diff_min)
    return diff_min_list



In [ ]:
sweep_times = get_runtimes(sweep_paths)
bgs_times = get_runtimes(bgs_paths)
neutral_times = get_runtimes(neutral_paths)

In [ ]:
bgs_times.keys()

In [ ]:
sweep_times.keys()

In [ ]:
neutral_times.keys()

In [23]:
neutral_times_arr = np.array(neutral_times[""])
bgs_times_arr = np.array(bgs_times["annot=ensembl_havana_104_exons, dfe=Gamma_K17,"])
sweep_times_yri_bgs_arr = np.array(sweep_times["popu=YRI, annot=ensembl_havana_104_exons, dfe=Gamma_K17, coeff=0.03, tmult=1,"])
sweep_times_ceu_bgs_arr = np.array(sweep_times["popu=CEU, annot=ensembl_havana_104_exons, dfe=Gamma_K17, coeff=0.03, tmult=1,"])
sweep_times_yri_nobgs_arr = np.array(sweep_times["popu=YRI, annot=NA, dfe=NA, coeff=0.03, tmult=1,"])
sweep_times_ceu_nobgs_arr = np.array(sweep_times["popu=CEU, annot=NA, dfe=NA, coeff=0.03, tmult=1,"])



In [ ]:
print("Number of files:")
print(len(sweep_paths))
print(len(bgs_paths))
print(len(neutral_paths))

print("Number of runtimes:")
print(len(sweep_times_yri_bgs_arr))
print(len(sweep_times_ceu_bgs_arr))
print(len(sweep_times_yri_nobgs_arr))
print(len(sweep_times_ceu_nobgs_arr))
print(len(bgs_times_arr))
print(len(neutral_times_arr))


In [ ]:
fig, axs = plt.subplots(6, 1, sharex=True, tight_layout=True, figsize=(8.5,11))

n_bins = 100
axs[0].hist(sweep_times_yri_bgs_arr, bins=n_bins, label="Sweep in YRI with BGS")
axs[1].hist(sweep_times_ceu_bgs_arr, bins=n_bins, label="Sweep in CEU with BGS")
axs[2].hist(bgs_times_arr, bins=n_bins, label="BGS")
axs[3].hist(sweep_times_yri_nobgs_arr, bins=n_bins, label="Sweep in YRI")
axs[4].hist(sweep_times_ceu_nobgs_arr, bins=n_bins, label="Sweep in CEU")
axs[5].hist(neutral_times_arr, bins=n_bins, label="Neutral")

axs[0].set_title("Sweep in YRI with BGS")
axs[1].set_title("Sweep in CEU with BGS")
axs[2].set_title("BGS")
axs[3].set_title("Sweep in YRI")
axs[4].set_title("Sweep in CEU")
axs[5].set_title("Neutral")

axs[5].set_xlabel("Runtime (minutes)")

plt.savefig("results/sweep_runtime_distributions.pdf")

plt.show()

In [ ]:
sweep_paths = np.array(sweep_paths)
sweep_times = np.array(sweep_times)
sweep_paths[sweep_times<1]

In [ ]:
np.nanmean(subsampled_sweep_times)